# 3.9 Pivot Tablolar

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/09-pivot-tables.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Pivot Tables

groupby soyutlamasının veri kümesi içindeki ilişkileri keşfetmeye nasıl yardımcı olduğunu gördük. Pivot tablo, elektronik tablolarda ve tablo verisiyle çalışan programlarda yaygın benzer bir işlemdir: sütun yönelimli veriyi alır, girişleri verinin çok boyutlu özetini veren iki boyutlu bir tabloda gruplar. Pivot tablolar ile groupby bazen karıştırılır; pivot tabloyu groupby agregasyonunun çok boyutlu sürümü olarak düşünmek yardımcı olur — böl ve birleştir hem tek boyutlu indeks yerine iki boyutlu ızgarada olur.

## Pivot Tablolara Giriş

Bu bölümdeki örnekler için Seaborn üzerinden Titanic yolcu veri tabanını kullanacağız:


In [ ]:
# import_titanic.py
import numpy as np
import pandas as pd
import seaborn as sns
titanic = sns.load_dataset('titanic')



In [ ]:
# titanic_head.py
titanic.head()



Çıktıda, felaketli yolculuktaki her yolcu için cinsiyet, yaş, sınıf, ödenen ücret ve daha fazlası yer alır.

## Elle Pivot Tablo

Veriyi anlamak için cinsiyet, hayatta kalma durumu veya bunların birleşimine göre gruplamak isteyebiliriz. Önceki bölümü okuduysanız groupby kullanmaya meyillisiniz — cinsiyete göre hayatta kalma oranı:


In [ ]:
# titanic_survival_sex.py
titanic.groupby('sex')[['survived']].mean()



İlk içgörü: gemideki kadınların dörtte üçü, erkeklerin ise yaklaşık beşte biri hayatta kalmış!

Bir adım daha gidip hem cinsiyet hem sınıfa göre hayatta kalma oranına bakalım. groupby sözcüğüyle: sınıf ve cinsiyete göre grupla, hayatta kalmayı seç, ortalama uygula, grupları birleştir, hiyerarşik indeksi unstack ile aç:


In [ ]:
# titanic_groupby_unstack.py
titanic.groupby(['sex', 'class'])['survived'].aggregate('mean').unstack()



Cinsiyet ve sınıfın hayatta kalmayı nasıl etkilediğine dair daha iyi bir fikir verir; kod ise okunması zor bir dizi haline gelmeye başlar. Bu iki boyutlu groupby yeterince yaygındır; Pandas pivot_table adlı kısa yolu içerir.

## Pivot Tablo Sözdizimi

Önceki işlemin DataFrame.pivot_table ile eşdeğeri:


In [ ]:
# titanic_pivot_table.py
titanic.pivot_table('survived', index='sex', columns='class', aggfunc='mean')



Elle groupby yaklaşımından çok daha okunaklıdır; aynı sonucu üretir. 20. yüzyıl başı transatlantik yolculuğunda hayatta kalma eğilimi hem üst sınıfları hem de veride kadın olarak kayıtlı yolcuları kayırır. Birinci sınıf kadınlar neredeyse kesin hayatta kalmış (merhaba Rose!); üçüncü sınıf erkeklerin yaklaşık sekizde biri (üzgünüz Jack!).

### Çok Düzeyli Pivot Tablolar

groupby'da olduğu gibi pivot tablolarda gruplama birden fazla düzeyle ve seçeneklerle belirtilebilir. Yaşı üçüncü boyut olarak ekleyelim; pd.cut ile kutulayalım:


In [ ]:
# titanic_pivot_age.py
age = pd.cut(titanic['age'], [0, 18, 80])
titanic.pivot_table('survived', ['sex', age], 'class')



Sütunlar için de aynı strateji: ücret için pd.qcut ile otomatik niceleme:


In [ ]:
# titanic_pivot_fare.py
fare = pd.qcut(titanic['fare'], 2)
titanic.pivot_table('survived', ['sex', age], [fare, 'class'])



Sonuç, değerler arasındaki ilişkiyi ızgara düzeninde gösteren hiyerarşik indeksli (3.5 Hiyerarşik İndeksleme) dört boyutlu bir agregasyondur.

### Ek Pivot Tablo Seçenekleri

DataFrame.pivot_table tam imzası (Pandas 1.3.5):


```python
# Pandas 1.3.5 imzası
DataFrame.pivot_table(data, values=None, index=None, columns=None,
                      aggfunc='mean', fill_value=None, margins=False,
                      dropna=True, margins_name='All', observed=False,
                      sort=True)
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


İlk üç argümanı gördük; fill_value ve dropna eksik veriyle ilgilidir.

aggfunc hangi agregasyonun uygulanacağını denetler (varsayılan ortalama). groupby'da olduğu gibi dize ('sum', 'mean', 'count' vb.) veya fonksiyon olabilir; sütunlara eşleme sözlüğü de verilebilir:


In [ ]:
# titanic_pivot_aggfunc_dict.py
titanic.pivot_table(index='sex', columns='class',
                    aggfunc={'survived':sum, 'fare':'mean'})



aggfunc için eşleme verildiğinde values anahtar sözcüğü çoğu zaman otomatik belirlenir.

Bazen her gruplama boyunca toplamlar yararlıdır; margins anahtar sözcüğü ile:


In [ ]:
# titanic_pivot_margins.py
titanic.pivot_table('survived', index='sex', columns='class', margins=True)



Bu, sınıftan bağımsız cinsiyete göre hayatta kalma, cinsiyetten bağımsız sınıfa göre hayatta kalma ve genel %38 hayatta kalma oranını verir. Kenar etiketi margins_name ile değiştirilir (varsayılan "All").

## Örnek: Doğum Oranı Verisi

Başka bir örnek: ABD Hastalık Kontrol Merkezleri (CDC) doğum verisi. Veri burada bulunur (Andrew Gelman ve grubu tarafından kapsamlı analiz edilmiştir).


```
# shell command to download the data:
# !cd data && curl -O \
# https://raw.githubusercontent.com/jakevdp/data-CDCbirths/master/births.csv
```


In [ ]:
# read_births_csv.py
births = pd.read_csv('data/births.csv')



> **Not**
>

Veri görece basit — tarih ve cinsiyete göre gruplanmış doğum sayıları.


In [ ]:
# births_head.py
births.head()



decade sütunu ekleyip on yıllara göre erkek/kadın doğumlarına pivot tablo ile bakalım:


In [ ]:
# births_pivot_decade.py
births['decade'] = 10 * (births['year'] // 10)
births.pivot_table('births', index='decade', columns='gender', aggfunc='sum')



Her on yılda erkek doğumları kadın doğumlarından fazladır. Pandas yerleşik plot ile yıllık trend görselleştirilebilir (Matplotlib bölümüne bakın):


```
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
births.pivot_table(
    'births', index='year', columns='gender', aggfunc='sum').plot()
plt.ylabel('total births per year');
```


Basit pivot tablo ve plot ile cinsiyete göre yıllık doğum eğilimi hemen görülür; son 50 yılda erkek doğumları kadın doğumlardan yaklaşık %5 fazla gibi görünür.

Pivot tabloyla doğrudan ilgili olmasa da bu veri kümesinden Pandas araçlarıyla birkaç ilginç özellik daha çıkarılabilir. Önce veriyi temizleyelim — yanlış yazılmış tarihler (ör. 31 Haziran) veya eksik günler (99 Haziran) gibi aykırıları kırpma (robust sigma-clipping) ile kaldıralım:


In [ ]:
# births_quartiles.py
quartiles = np.percentile(births['births'], [25, 50, 75])
mu = quartiles[1]
sig = 0.74 * (quartiles[2] - quartiles[0])



Son satır, 0,74 çarpanı Gauss dağılımının çeyreklikler aralığından gelen örneklem standart sapmasının sağlam bir tahminidir.

Bu değerlerle query (3.12 eval ve query) kullanarak aykırı satırları filtreleyebiliriz:


In [ ]:
# births_query_clip.py
births = births.query('(births > @mu - 5 * @sig) & (births < @mu + 5 * @sig)')



day sütununu tamsayı yapalım; bazı satırlarda 'null' olduğu için önceden metindi:


In [ ]:
# set 'day' column to integer; it originally was a string due to nulls
births['day'] = births['day'].astype(int)



Son olarak gün, ay ve yılı birleştirip tarih indeksi oluşturalım (3.11 Zaman Serileri):


In [ ]:
# create a datetime index from the year, month, day
births.index = pd.to_datetime(10000 * births.year +
                              100 * births.month +
                              births.day, format='%Y%m%d')

births['dayofweek'] = births.index.dayofweek



Bununla on yıllar boyunca haftanın gününe göre doğumları çizebiliriz:


In [ ]:
# births_weekday_plot.py
import matplotlib.pyplot as plt
import matplotlib as mpl

births.pivot_table('births', index='dayofweek',
                    columns='decade', aggfunc='mean').plot()
plt.gca().set(xticks=range(7),
              xticklabels=['Mon', 'Tues', 'Wed', 'Thurs', 'Fri', 'Sat', 'Sun'])
plt.ylabel('mean births by day');



Görünüşe göre hafta sonları doğumlar hafta içinden biraz daha az! 1990 ve 2000'ler eksik — 1989'dan itibaren CDC verisi yalnızca doğum ayını içeriyor.

İlginç bir başka görünüm: yılın gününe göre ortalama doğum sayısı. Önce ay ve güne göre gruplayalım:


In [ ]:
# births_by_date_pivot.py
births_by_date = births.pivot_table('births', 
                                    [births.index.month, births.index.day])
births_by_date.head()



Sonuç ay ve gün üzerinde çoklu indekstir. Görselleştirmek için ay ve günleri şubat 29'u doğru işleyen artık yıl ile birleştirip tarihe dönüştürelim:


In [ ]:
# births_by_date_index.py
from datetime import datetime
births_by_date.index = [datetime(2012, month, day)
                        for (month, day) in births_by_date.index]
births_by_date.head()



Yalnızca ay ve güne odaklanınca yılın gününe göre ortalama doğum sayısını yansıtan bir zaman serimiz olur. plot ile çizildiğinde ilginç eğilimler görülür:


In [ ]:
# Plot the results
fig, ax = plt.subplots(figsize=(12, 4))
births_by_date.plot(ax=ax);



> **Not**
>

Özellikle grafikte ABD tatillerinde doğum oranı düşüşü çarpıcıdır — büyük olasılıkla doğal doğumdan çok planlı/indüklenmiş doğum eğilimlerini yansıtır. Konuyla ilgili Andrew Gelman'ın blog yazısına bakın. Bu grafiğe Matplotlib araçlarıyla not eklemek için kitabın ilerleyen bölümlerine bakılacaktır.

Bu kısa örnek, şimdiye kadar gördüğümüz Python ve Pandas araçlarının birleştirilerek çeşitli veri kümelerinden içgörü elde edilebileceğini gösterir. Bu veri manipülasyonlarının daha gelişmiş uygulamalarını sonraki bölümlerde göreceğiz!

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Titanic verisinde (Seaborn yüklüyse) cinsiyet × sınıf hayatta kalma pivot tablosu oluşturun:
          
      import pandas as pd
try:
    import seaborn as sns
    t = sns.load_dataset('titanic')
    print(t.pivot_table('survived', index='sex', columns='class', aggfunc='mean'))
except Exception:
    df = pd.DataFrame({
        'sex': ['female','female','male','male'],
        'class': ['First','Third','First','Third'],
        'survived': [1, 0, 1, 0]
    })
    print(df.pivot_table('survived', index='sex', columns='class', aggfunc='mean'))

> **Not**
>
